**Part 1**

Eleanor Warren

ST 554 (601)

Testing class on data. First, import necessary modules, create a spark session, and import my .py file to define my class.

In [1]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
import pyspark.sql.types as T
import pandas as pd

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config('spark.sql.ansi.enabled', 'false').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 21:11:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import Project2_Part1_WarrenE

In [4]:
# Commenting out but keeping in for final project - useful to reload class code again
#import importlib
#importlib.reload(Project2_Part1_WarrenE)

Next, I want to create an instance of my class using the method that takes in data from a .csv file. A few things to note about this step:

- I go ahead and create two string variables `over100` and `cold`, in order to have some columns to test my string-related methods on later in this file.
- I recode -200 in the data with "None"
- I drop the column `_c0` which was the automatic name given to a column with no header, that appeared to contain a row number/ID. Keeping this column in caused warnings to appear, because the automatic schema that was defined based on the header didn't match the actual .csv headers, in which this column is blank. We don't need it for our analysis anyway.
- I rename most of the columns to names that are easier to use and won't cause syntax incompatibilities

In [5]:
csv_data = 'air.csv'
my_instance = Project2_Part1_WarrenE. \
        SparkDataCheck.from_csv(spark, csv_data)
my_instance.df = my_instance.df \
        .withColumn("over100", \
        F.when(my_instance.df["NMHC(GT)"] > 100, "over 100") \
        .otherwise("less/equal to 100")) \
        .withColumn("cold", \
        F.when(my_instance.df["T"] < 13, "cold") \
        .otherwise("not cold")) \
        .replace(-200, None) \
        .drop("_c0") \
        .withColumnsRenamed({"CO(GT)": "CO", \
                             "PT08.S1(CO)": "S1", \
                             "NMHC(GT)": "NMHC", \
                             "C6H6(GT)": "C6H6", \
                             "PT08.S2(NMHC)": "S2", \
                             "NOx(GT)": "NOX", \
                             "PT08.S3(NOx)": "S3", \
                             "NO2(GT)": "NO2", \
                             "PT08.S4(NO2)": "S4", \
                             "PT08.S5(O3)": "S5"})

Since I've redefined the df attribute on my class instance, I can check both the columns and schema to make sure it worked how I wanted it to.

In [6]:
my_instance.df.columns

['Date',
 'Time',
 'CO',
 'S1',
 'NMHC',
 'C6H6',
 'S2',
 'NOX',
 'S3',
 'NO2',
 'S4',
 'S5',
 'T',
 'RH',
 'AH',
 'over100',
 'cold']

In [7]:
my_instance.df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO: double (nullable = true)
 |-- S1: integer (nullable = true)
 |-- NMHC: integer (nullable = true)
 |-- C6H6: double (nullable = true)
 |-- S2: integer (nullable = true)
 |-- NOX: integer (nullable = true)
 |-- S3: integer (nullable = true)
 |-- NO2: integer (nullable = true)
 |-- S4: integer (nullable = true)
 |-- S5: integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)
 |-- over100: string (nullable = false)
 |-- cold: string (nullable = false)



Now, I'll start testing out my methods on the air data. First, let's do some examples with `.check_limits()`, which tests whether a numeric variable's values fall within a specified range. It appends the Boolean `inBounds` to our dataframe. The examples below are for the scenarios:

- When no bounds are given
- When only an upper bound is given
- When only a lower bound is given
- When both bounds are given

In [8]:
my_instance.check_limits("CO", None, None).df.show(3)

No upper or lower bounds provided
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
only showing top 3 rows


In [9]:
my_instance.check_limits("CO", None, 10).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


In [10]:
my_instance.check_limits("CO", 1.5, None).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


In [11]:
my_instance.check_limits("CO", 1.5, 4).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


Next is the method `.check_string()`, which tests if a string variable's values are found within a list of string "levels". It appends the Boolean column `inLevels` to our dataframe. The scenarios are:

- A string variable tested against a string level
- A string variable tested against multiple string levels
- A numeric variable tested against a string level
- Another string variable tested against a string level

In [12]:
my_instance.check_string("over100", ["over 100"]).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
only showing 

In [13]:
my_instance.check_string("over100", ["over 100", "less/equal to 100"]).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
only showing 

In [14]:
my_instance.check_string("CO", ["over 100"]).df.show(3)

Supplied column is not string type. No changes have been made to the dataframe.
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+-

In [15]:
my_instance.check_string("cold", ["hot"]).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
only showing 

Next, we'll test the `.check_missing()` method, which appends a Boolean `isMissing` to our dataframe, and tests whether or not the values in the supplied column are missing. The scenarios below just check different columns in our dataframe.

In [16]:
my_instance.check_missing("CO").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [17]:
my_instance.check_missing("T").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [18]:
my_instance.check_missing("RH").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [19]:
my_instance.check_missing("Date").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

Below we have the `.get_minmax()` method, which prints a pandas dataframe of the minimum and maximum values for the numeric columns supplied. The scenarios are: 

- Min/max for one numeric column
- Min/max for a numeric column grouped by a string column
- Min/max error for a string column
- Min/max for all numeric columns (pass None)
- Min/max for all numeric columns grouped by a string

In [20]:
my_instance.get_minmax("RH")

,min(RH),max(RH)
0,9.2,88.7


In [21]:
my_instance.get_minmax("RH", "over100")

,over100,min(RH),max(RH)
0,less/equal to 100,9.2,88.7
1,over 100,14.9,81.8


In [22]:
my_instance.get_minmax("Date")

Error: The supplied column is not numeric type


In [23]:
my_instance.get_minmax(None)

,min(CO),max(CO),min(S1),max(S1),min(NMHC),max(NMHC),min(C6H6),max(C6H6),min(S2),max(S2),...,min(S4),max(S4),min(S5),max(S5),min(T),max(T),min(RH),max(RH),min(AH),max(AH)
0,0.1,11.9,647,2040,7,1189,0.1,63.7,383,2214,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


In [24]:
my_instance.get_minmax(None, "cold")

,cold,min(CO),max(CO),cold,min(S1),max(S1),cold,min(NMHC),max(NMHC),cold,...,max(S5),cold,min(T),max(T),cold,min(RH),max(RH),cold,min(AH),max(AH)
0,not cold,0.1,10.2,not cold,667,2040,not cold,9,1189,not cold,...,2519,not cold,13.0,44.6,not cold,9.2,87.2,not cold,0.2185,2.2310
1,cold,0.1,11.9,cold,647,2008,cold,7,1042,cold,...,2523,cold,-1.9,12.9,cold,13.5,88.7,cold,0.1847,1.2422


Lastly, we have the method `.get_stringlevels()` which provides a count for each unique level of a string variable or combination of two string variables. The scenarios are:

- counts for a string column
- counts for another string column
- counts for the combinations for two string columns
- Attempting to pass a string and numeric
- Attempting to pass a numeric and string

In [25]:
my_instance.get_stringlevels("over100")

+-----------------+-----+
|          over100|count|
+-----------------+-----+
|less/equal to 100| 8787|
|         over 100|  570|
+-----------------+-----+



In [26]:
my_instance.get_stringlevels("cold")

+--------+-----+
|    cold|count|
+--------+-----+
|not cold| 6317|
|    cold| 3040|
+--------+-----+



In [27]:
my_instance.get_stringlevels("over100", "cold")

+-----------------+--------+-----+
|          over100|    cold|count|
+-----------------+--------+-----+
|less/equal to 100|    cold| 2912|
|less/equal to 100|not cold| 5875|
|         over 100|not cold|  442|
|         over 100|    cold|  128|
+-----------------+--------+-----+



In [28]:
my_instance.get_stringlevels("over100", "CO")

In [29]:
my_instance.get_stringlevels("CO", "cold")

Error: All supplied columns are not StringType. Please supply a string column.


Now, create an instance of our class from a pandas dataframe. We read the csv in and save it as a pandas dataframe, then pass it to our class statement. I do the same data cleaning for this SQL dataframe as in the first example.

In [30]:
# Get a regular pandas dataframe from our csv
pdf = pd.read_csv("air.csv")

# Create an instance of our class from the pandas dataframe
pd_instance = Project2_Part1_WarrenE.SparkDataCheck.from_pandas(spark, pdf)

# Clean our instance dataframe as above
pd_instance.df = pd_instance.df \
        .withColumn("over100", \
        F.when(pd_instance.df["NMHC(GT)"] > 100, "over 100") \
        .otherwise("less/equal to 100")) \
        .withColumn("cold", \
        F.when(pd_instance.df["T"] < 13, "cold") \
        .otherwise("not cold")) \
        .replace(-200, None) \
        .drop("_c0") \
        .withColumnsRenamed({"CO(GT)": "CO", \
                             "PT08.S1(CO)": "S1", \
                             "NMHC(GT)": "NMHC", \
                             "C6H6(GT)": "C6H6", \
                             "PT08.S2(NMHC)": "S2", \
                             "NOx(GT)": "NOX", \
                             "PT08.S3(NOx)": "S3", \
                             "NO2(GT)": "NO2", \
                             "PT08.S4(NO2)": "S4", \
                             "PT08.S5(O3)": "S5"})

Show that we can also use the same methods on this instance of our class.

In [31]:
pd_instance.get_minmax("NMHC")

,min(NMHC),max(NMHC)
0,7,1189


**Part 2**

Load in our csv through regular pandas, then convert to Pandas-On-Spark and show the first 5 rows.

In [32]:
import pyspark.pandas as ps
import numpy as np

pdf = pd.read_csv("weekly_nfl_data.csv", delimiter = ",")
psdf = ps.from_pandas(pdf)
psdf.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
26/03/29 21:11:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Display our Pandas on Spark dataframe columns.

In [33]:
psdf.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

Subset our dataframe to only keep the columns listed in `keep_cols`, subsetted based on the given conditions.

In [34]:
# Make a list of columns to keep in our subsetted data
keep_cols = ["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]

# Subset data based on the following row conditions
psdf = psdf.loc[(psdf["position"] == "QB") & (psdf["season_type"] == "REG") & (psdf["season"].between(2005,2023)), keep_cols]
psdf.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


Find the average of all the columns after grouping by player name and season, then show the first few rows.

In [35]:
psdf.groupby(["player_display_name", "season"]).mean().head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions
player_display_name,season,,,,,,
Jeff Blake,2005,9.500000,4.000000,4.500000,27.500000,0.500000,0.000000
Daunte Culpepper,2005,4.428571,19.857143,30.857143,223.428571,0.857143,1.714286
Kerry Collins,2005,8.933333,20.133333,37.733333,250.600000,1.333333,0.866667
Tony Banks,2005,17.000000,14.000000,25.000000,173.000000,1.000000,2.000000
Charlie Batch,2005,11.666667,7.666667,12.000000,82.000000,0.333333,0.333333


Find the sum of all the columns after grouping by player name and season, then show the first few rows.

In [36]:
psdf.groupby(["player_display_name", "season"]).sum().head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions
player_display_name,season,,,,,,
Jeff Blake,2005,19,8,9,55.0,1,0.0
Daunte Culpepper,2005,31,139,216,1564.0,6,12.0
Kerry Collins,2005,134,302,566,3759.0,20,13.0
Tony Banks,2005,17,14,25,173.0,1,2.0
Charlie Batch,2005,35,23,36,246.0,1,1.0


Group by player name and season, find the sum of all columns, then use that to define two new variables `completion_percentage` and `td_int_ratio`.

In [37]:
collapse = psdf.groupby(["player_display_name", "season"]).sum()
#.rename(columns = {"age": "o_age", "fare": "o_fare"}

collapse["completion_percentage"] =  collapse["completions"] / collapse["attempts"]
collapse["td_int_ratio"] = collapse["passing_tds"]/collapse["interceptions"]
collapse.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Jeff Blake,2005,19,8,9,55.0,1,0.0,0.888889,inf
Daunte Culpepper,2005,31,139,216,1564.0,6,12.0,0.643519,0.500000
Kerry Collins,2005,134,302,566,3759.0,20,13.0,0.533569,1.538462
Tony Banks,2005,17,14,25,173.0,1,2.0,0.560000,0.500000
Charlie Batch,2005,35,23,36,246.0,1,1.0,0.638889,1.000000


Filter our grouped dataset by count of attempts greater or equal to 50.

In [38]:
collapse = collapse.loc[collapse["attempts"] >= 50]

Sort the dataframe in descending order by `completion_percentage`.

In [39]:
collapse \
  .sort_values(by = "completion_percentage", ascending = False) \
  .head(40)

week  completions  attempts  passing_yards  passing_tds  interceptions  completion_percentage  td_int_ratio
player_display_name season                                                                                                             
C.J. Beathard       2023      65           40        53          349.0            1            0.0               0.754717           inf
Colt McCoy          2021      62           74        99          740.0            3            1.0               0.747475      3.000000
Matt Schaub         2019      52           50        67          580.0            3            1.0               0.746269      3.000000
Drew Brees          2018     130          364       489         3992.0           32            5.0               0.744376      6.400000
                    2019     119          281       378         2979.0           27            4.0               0.743386      6.750000
Mason Rudolph       2023      66           55        74          719.0            3            0.0               0.743243           inf
Taysom Hill         2020     147           88       121          928.0            4            2.0               0.727273      2.000000
Nick Foles          2018      51          141       195         1413.0            7            4.0               0.723077      1.750000
Drew Brees          2017     148          386       536         4334.0           23            8.0               0.720149      2.875000
Sam Bradford        2016     146          395       552         3877.0           20            5.0               0.715580      4.000000
Drew Brees          2011     142          471       660         5535.0           46           14.0               0.713636      3.285714
Colt McCoy          2014      57           91       128         1057.0            4            3.0               0.710938      1.333333
Aaron Rodgers       2020     148          372       526         4299.0           48            5.0               0.707224      9.600000
Bailey Zappe        2022      22           65        92          781.0            5            3.0               0.706522      1.666667
Drew Brees          2009     131          363       514         4388.0           34           11.0               0.706226      3.090909
                    2020      97          275       390         2942.0           24            6.0               0.705128      4.000000
Joe Burrow          2021     143          366       520         4611.0           34           14.0               0.703846      2.428571
Derek Carr          2019     147          361       513         4054.0           21            8.0               0.703704      2.625000
Jake Browning       2023     117          171       243         1936.0           12            7.0               0.703704      1.714286
Chase Daniel        2019      20           45        64          435.0            3            2.0               0.703125      1.500000
Ryan Tannehill      2019     128          201       286         2742.0           22            6.0               0.702797      3.666667
Deshaun Watson      2020     145          382       544         4823.0           33            7.0               0.702206      4.714286
Alex Smith          2012      63          153       218         1737.0           13            5.0               0.701835      2.600000
Kirk Cousins        2018     143          425       606         4298.0           30           10.0               0.701320      3.000000
Jamie Martin        2005      92          124       177         1277.0            5            7.0               0.700565      0.714286
Drew Brees          2016     148          471       673         5208.0           37           15.0               0.699851      2.466667
Tony Romo           2014     133          304       435         3705.0           34            9.0               0.698851      3.777778
Matt Ryan           2016     142          373       534         4944.0           38 

 Sort in descending order by `td_int_ratio`. Note that we have "inf" values where `td_int_ratio` is divided by 0.

In [40]:
collapse \
  .sort_values(by = "td_int_ratio", ascending = False) \
  .head(40)

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Matt Schaub,2005,65,33,64,495.0,4,0.0,0.515625,inf
Charlie Batch,2006,71,30,52,477.0,5,0.0,0.576923,inf
Todd Collins,2007,62,67,105,888.0,5,0.0,0.638095,inf
Troy Smith,2007,62,40,76,452.0,2,0.0,0.526316,inf
Jake Locker,2011,51,34,66,542.0,4,0.0,0.515152,inf
Derek Anderson,2014,30,65,97,701.0,5,0.0,0.670103,inf
Brian Hoyer,2016,27,134,200,1445.0,6,0.0,0.670000,inf
Nick Foles,2016,17,36,55,410.0,3,0.0,0.654545,inf
Jimmy Garoppolo,2016,49,43,63,502.0,4,0.0,0.682540,inf


Now, we want to recreate this process but from a Spark SQL DataFrame.

In [41]:
sqdf = spark.createDataFrame(pdf)
sqdf.show()

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+-----------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|

In [42]:
sqdf.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

Subset our dataframe based on the same criteria.

In [43]:
# Subset data based on the following row conditions
sqdf = sqdf.where((sqdf["position"] == "QB") \
            & (sqdf["season_type"] == "REG") \
            & (sqdf["season"].between(2005,2023))) \
            .select(["player_display_name", \
            "season", \
            "week", \
            "completions", \
            "attempts", \
            "passing_yards", \
            "passing_tds", \
            "interceptions"]) \
            .where((sqdf["position"] == "QB") \
            & (sqdf["season_type"] == "REG") \
            & (sqdf["season"].between(2005,2023)))

In [44]:
sqdf.groupBy(["player_display_name", "season"]).sum().show(3)

+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
|player_display_name|season|sum(season)|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|
+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
|         Jeff Blake|  2005|       4010|       19|               8|            9|              55.0|               1|               0.0|
|   Daunte Culpepper|  2005|      14035|       31|             139|          216|            1564.0|               6|              12.0|
|      Kerry Collins|  2005|      30075|      134|             302|          566|            3759.0|              20|              13.0|
+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
only showing top 3 rows


In [45]:
sqdf.groupBy(["player_display_name", "season"]).mean().show(3)

+-------------------+------+-----------+-----------------+------------------+------------------+------------------+------------------+------------------+
|player_display_name|season|avg(season)|        avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+-----------+-----------------+------------------+------------------+------------------+------------------+------------------+
|         Jeff Blake|  2005|     2005.0|              9.5|               4.0|               4.5|              27.5|               0.5|               0.0|
|   Daunte Culpepper|  2005|     2005.0|4.428571428571429|19.857142857142858|30.857142857142858|223.42857142857142|0.8571428571428571|1.7142857142857142|
|      Kerry Collins|  2005|     2005.0|8.933333333333334|20.133333333333333|37.733333333333334|             250.6|1.3333333333333333|0.8666666666666667|
+-------------------+------+-----------+-----------------+------------------

In [46]:
scollapse = sqdf.groupBy(["player_display_name", "season"]).sum()
#.rename(columns = {"age": "o_age", "fare": "o_fare"}

scollapse = scollapse \
        .withColumn("completion_percentage", \
        F.col("sum(completions)") / F.col("sum(attempts)")) \
        .withColumn("td_int_ratio", F.col("sum(passing_tds)") / F.col("sum(interceptions)"))

In [47]:
scollapse.filter(scollapse["sum(attempts)"] >= 50)

DataFrame[player_display_name: string, season: bigint, sum(season): bigint, sum(week): bigint, sum(completions): bigint, sum(attempts): bigint, sum(passing_yards): double, sum(passing_tds): bigint, sum(interceptions): double, completion_percentage: double, td_int_ratio: double]

In [48]:
scollapse.sort(scollapse.completion_percentage, ascending = False).show(40)

+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+---------------------+------------+
|player_display_name|season|sum(season)|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|completion_percentage|td_int_ratio|
+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+---------------------+------------+
|       Chase Daniel|  2015|       4030|       23|               2|            2|               4.0|               0|               0.0|                  1.0|        NULL|
|     Anthony Wright|  2006|       4012|       26|               3|            3|              31.0|               0|               0.0|                  1.0|        NULL|
|     Matt Gutierrez|  2007|       8028|       18|               1|            1|              15.0|               0|               0.0|    

Note that `td_int_ratio` is now treated as NULL wherever we divided by 0.

In [49]:
scollapse.sort(scollapse.td_int_ratio, ascending = False).show(40)

+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+---------------------+-----------------+
|player_display_name|season|sum(season)|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|completion_percentage|     td_int_ratio|
+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+---------------------+-----------------+
|          Tom Brady|  2016|      24192|      134|             291|          432|            3554.0|              28|               2.0|   0.6736111111111112|             14.0|
|         Nick Foles|  2013|      26169|      129|             203|          317|            2891.0|              27|               2.0|   0.6403785488958991|             13.5|
|        Josh McCown|  2013|      16104|       92|             149|          224|            1829.0|              1